# LinkedIn Zip Solver Setup Notebook

This notebook helps verify the repository modules and environment setup for `linkedin_zip_solver`.

Use the cells below to inspect imports, check the Python environment, and run the solver pipeline in a lightweight form.


In [7]:
# linkedin_zip_solver setup notebook

print('Notebook ready to run')


Notebook ready to run


## Quick checks

Use the cell below to verify the Python environment and repository imports.


In [8]:
import sys
import os
from pathlib import Path

print('Python executable:', sys.executable)
print('Python version:', sys.version)
print('Working directory:', Path.cwd())
print('Repository files:', sorted(p.name for p in Path('.').iterdir() if p.is_file()))


Python executable: c:\Users\shiva\Documents\Zip\linkedin_zip_solver\.venv\Scripts\python.exe
Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Working directory: c:\Users\shiva\Documents\Zip\linkedin_zip_solver
Repository files: ['README.md', 'append_cell.py', 'body_screenshot.png', 'cells.txt', 'check_classes.py', 'cropped.png', 'dump_cells.py', 'executor.py', 'fix_cell29.py', 'fix_cell29_output.py', 'fix_cell29_subproc.py', 'fix_nb.py', 'full_dom.html', 'inspect_dom.py', 'inspect_dom2.py', 'inspect_dom3.py', 'inspect_dom4.py', 'inspect_dom5.py', 'main.py', 'model.py', 'package.py', 'parse_dom.py', 'parsed.png', 'patch_vision.py', 'requirements.txt', 'setup_env.py', 'setup_notebook.ipynb', 'solver.py', 'test_crop.py', 'test_cropped_ocr.py', 'test_final.py', 'test_hough.py', 'test_hough2.py', 'test_idx.py', 'test_ocr.py', 'test_ratio.py', 'test_vision.py', 'train.py', 'vision.py', 'zip_cnn.pth']


In [9]:
from pathlib import Path
import importlib.util

print('\nModule import check:')
for module_name in ['main', 'solver', 'vision', 'executor', 'model', 'package', 'setup_env']:
    spec = importlib.util.find_spec(module_name)
    print(f'  {module_name}:', 'available' if spec is not None else 'missing')



Module import check:
  main: available
  solver: available
  vision: available
  executor: available
  model: available
  package: available
  setup_env: available


## Lightweight pipeline run

Use this cell to import the repo and print the CLI help. This avoids launching the browser but checks that the main orchestration module is importable.


In [10]:
import sys
from pathlib import Path

repo_root = Path.cwd()
sys.path.insert(0, str(repo_root))

import main

print('Imported main module successfully')
try:
    main.parse_args(['--help'])
except SystemExit as exc:
    print('main.parse_args(["--help"]) exited with code', exc.code)


Imported main module successfully
usage: ipykernel_launcher.py [-h] [--skip-setup] [--solve-only]
                             [--package-only] [--headless] [--no-cnn]
                             [--max-iterations MAX_ITERATIONS]

LinkedIn Zip game autonomous solver.

options:
  -h, --help            show this help message and exit
  --skip-setup          Skip Phase 1 (environment setup).
  --solve-only          Run only Phases 2-4; skip packaging.
  --package-only        Run only Phase 5 (rebuild the zip).
  --headless            Run Chromium headless (requires prior login).
  --no-cnn              Disable the CNN prior - use plain DFS.
  --max-iterations MAX_ITERATIONS
                        Maximum number of DFS nodes to explore.
main.parse_args(["--help"]) exited with code 0


## Full runnable setup

This section ensures the notebook is running from the repository root, installs any missing dependencies, loads the model, and executes a minimal solver test that does not require the LinkedIn game UI.


In [11]:
import sys
from pathlib import Path
import subprocess
import json

repo_root = Path.cwd()
if repo_root.name != 'linkedin_zip_solver':
    raise RuntimeError('Notebook must be opened from the linkedin_zip_solver repo root.')

# Ensure repo root is on sys.path for imports.
sys.path.insert(0, str(repo_root))

print('Repo root:', repo_root)
print('Python executable:', sys.executable)
print('Python version:', sys.version)

# Install requirements if missing.
requirements = [
    'playwright>=1.40.0',
    'torch>=2.1.0',
    'torchvision>=0.16.0',
    'opencv-python>=4.8.0',
    'numpy>=1.24.0',
    'pillow>=10.0.0',
]

print('\nChecking installed packages...')
missing = []
for req in requirements:
    pkg = req.split('>=')[0]
    try:
        __import__(pkg)
    except ImportError:
        missing.append(req)

if missing:
    print('Missing packages detected:')
    for pkg in missing:
        print('  ', pkg)
    print('\nInstalling missing packages into the current environment...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('All required packages are installed.')

print('\nImporting repo modules...')
import main
import model
import solver
import vision
import executor
import setup_env
import package

print('Imports successful: main, model, solver, vision, executor, setup_env, package')

# Validate model inference.
print('\nRunning a model inference smoke test...')
from model import ZipCNN, load_model, NUM_CHANNELS
import torch

state_tensor = torch.zeros(1, NUM_CHANNELS, 8, 8)
model_obj = load_model()
policy, value = model_obj.predict(state_tensor)
print('Model loaded successfully.')
print('Policy probabilities:', policy.tolist())
print('Value score:', value)

# Minimal solver smoke test: create a trivial synthetic 4x4 board with a start, a goal, and no waypoints.
print('\nRunning solver smoke test...')
from vision import CellType, GameState, Cell
import numpy as np

h, w = 3, 3
grid = np.full((h, w), int(CellType.EMPTY), dtype=np.int32)
grid[0, 0] = int(CellType.START)
grid[-1, -1] = int(CellType.GOAL)

cells = [
    Cell(row=0, col=0, cell_type=int(CellType.START), pixel_center=(0, 0)),
    Cell(row=h-1, col=w-1, cell_type=int(CellType.GOAL), pixel_center=(0, 0)),
]
state = GameState(
    grid=grid,
    numbers=np.array([], dtype=np.int32),
    start=(0, 0),
    goal=(h - 1, w - 1),
    cells=cells,
    canvas_bbox=(0, 0, w * 10, h * 10),
    cell_size=10,
)

config = solver.SolverConfig(max_iterations=100_000, use_cnn_prior=True, verbose=True)
path = solver.solve(state, config=config, model=model_obj)
if path is None:
    raise RuntimeError('Solver smoke test failed: no path found.')
print('Solver found path length:', len(path))
print('Path:', path)


Repo root: c:\Users\shiva\Documents\Zip\linkedin_zip_solver
Python executable: c:\Users\shiva\Documents\Zip\linkedin_zip_solver\.venv\Scripts\python.exe
Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]

Checking installed packages...
Missing packages detected:
   opencv-python>=4.8.0
   pillow>=10.0.0

Installing missing packages into the current environment...

Importing repo modules...
Imports successful: main, model, solver, vision, executor, setup_env, package

Running a model inference smoke test...
Model loaded successfully.
Policy probabilities: [0.2183770388364792, 0.283180296421051, 0.2730973958969116, 0.22534531354904175]
Value score: 0.5635313391685486

Running solver smoke test...
[solver] start=(0, 0) goal=(2, 2) waypoints=[]
[solver] nodes_explored=9 prunes_conn=1 prunes_wp=0
Solver found path length: 9
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (1, 1), (0, 1), (0, 2), (1, 2), (2, 2)]


## Run the Live Solver

Run this cell to open up the browser, navigate to LinkedIn Zip, and solve it automatically using the loaded model!

In [12]:
import subprocess
import sys

print('Starting the autonomous solver in a separate process...')
print('Please wait for the browser window to appear.')

result = subprocess.run(
    [sys.executable, 'main.py', '--solve-only', '--skip-setup'],
    text=True,
    capture_output=True
)

print('\n--- OUTPUT ---')
print(result.stdout)
if result.stderr:
    print('\n--- ERRORS ---')
    print(result.stderr)
print('\nDone with exit code:', result.returncode)


Starting the autonomous solver in a separate process...
Please wait for the browser window to appear.

--- OUTPUT ---

=== Phase 2: Vision & state capture ===
[main] Vision error: Unexpected error during state capture: unsupported operand type(s) for //: 'str' and 'int'
[main] Phase 2 failed - aborting.


Done with exit code: 1
